In [ ]:
from ultralytics import YOLO
# from ultralytics import settings
# settings['tensorboard'] = True
# settings._save()


GPU_ID = 0  # index of the CUDA device to use (see `nvidia-smi` for available GPUs)

# model = YOLO("models/base/yolo26s-obb.pt") 
model = YOLO("last.pt ") 

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("./src/python"))

import albumentations as A
from src.python.spring_augment import register_spring_occlusion

IMG_SIZE = 720

# Aggressive augmentations layered on top of Ultralytics' built-in geometric/color jitter.
# Targets robustness to phone-camera artefacts: tight/partial crops, sensor noise, JPEG compression.
augmentations = [
    # A.RandomSizedBBoxSafeCrop(height=IMG_SIZE, width=IMG_SIZE, erosion_rate=0.2, p=0.3),
    # A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(0.05, 0.2), hole_width_range=(0.05, 0.2), p=0.3),
    A.ISONoise(p=0.2),
    A.ImageCompression(quality_range=(40, 90), p=0.06),
    A.Blur(blur_limit=3, p=0.1),
    A.MedianBlur(blur_limit=3, p=0.05),
    A.RandomBrightnessContrast(p=0.1),
    A.RandomGamma(p=0.2),
    A.ToGray(p=0.02),
    A.CLAHE(p=0.05),
]

# Shared with the augmentation-preview cell below so the preview matches training exactly.
train_kwargs = dict(
    data="./datasets/SKU110K_fixed/sku110k_obb.yaml",
    epochs=150,
    imgsz=IMG_SIZE,
    single_cls=True,
    degrees=10.0,    # Slightly rotate images
    flipud=0.5,      # Flip up-down
    fliplr=0.5,      # Flip left-righ
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    batch=8,
    augmentations=augmentations,
    device=GPU_ID,
    resume=True
)

# Synthetic vending-machine spring/auger occlusion (datasets/aug_springs), anchored to
# random object bboxes -- mimicking the coil dispenser arc that wraps across products
# on-shelf. Inserted directly into the dataset's transform pipeline (bypassing
# Albumentations) so it can read instance bboxes and place patches over real objects.
# `coverage` is the fraction of bboxes that get a spring patch when triggered.
spring_kwargs = dict(p=0.3, coverage=(0.05, 0.4), scale_range=(0.2, 0.8))
model.add_callback(
    "on_pretrain_routine_end",
    lambda trainer: register_spring_occlusion(trainer.train_loader.dataset, **spring_kwargs),
)


In [ ]:
# Preview the *exact* training-time pipeline (mosaic, affine/rotate/shear/perspective,
# mixup, the custom Albumentations transforms, HSV jitter, flips, spring occlusion) on
# real batches, without launching a full training run.
#
# Note: the first run will scan/cache all labels in datasets/SKU110K_fixed/labels/train
# (one-time cost, also speeds up the real training run afterwards).
import matplotlib.pyplot as plt
from ultralytics.cfg import DEFAULT_CFG, get_cfg
from ultralytics.data.build import build_dataloader, build_yolo_dataset
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils.plotting import plot_images

preview_batch = 4  # smaller than train batch so each image is easier to inspect
preview_cfg = get_cfg(DEFAULT_CFG, {**train_kwargs, "task": "obb", "mode": "train"})
data = check_det_dataset(preview_cfg.data)

dataset = build_yolo_dataset(preview_cfg, img_path=data["train"], batch=preview_batch, data=data, mode="train")
register_spring_occlusion(dataset, **spring_kwargs)
loader = build_dataloader(dataset, batch=preview_batch, workers=0, shuffle=True)

for i, batch in zip(range(3), loader):
    # show_labels=False: bbox text overlays the spring patches and makes them hard to see
    grid = plot_images(batch, names=data["names"], save=False, threaded=False, show_labels=False)
    plt.figure(figsize=(10, 10))
    plt.imshow(grid)
    plt.axis("off")
    plt.title(f"augmented batch {i}")
    plt.show()


In [ ]:
model.train(**train_kwargs)

In [ ]:
model.save("./models/tuned/items_detect.yolo26s-obb.pt")

In [ ]:
import glob
import os
import random
import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO("./models/tuned/items_detect.yolo26n-obb.pt")

print("loaded model")
test_images_path = "./datasets/SKU110K_fixed/images/test/*.jpg"  # Update path if needed
image_paths = glob.glob(test_images_path)

if not image_paths:
    raise FileNotFoundError("No images found. Please verify your test images path.")

# Select a random subset to display (e.g., 4 samples)
num_samples = min(4, len(image_paths))
sampled_paths = random.sample(image_paths, num_samples)

# 3. Setup Matplotlib grid (assuming a 2x2 grid for 4 samples)
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.ravel()  # Flatten the 2D array of axes into 1D for easy iteration

# 4. Infer and plot
for idx, img_path in enumerate(sampled_paths):
    # Run segmentation prediction
    results = model.predict(source=img_path, conf=0.25, verbose=False)
    
    # Extract the first result from the list
    result = results[0]
    

    annotated_img = result.plot(pil=True, boxes=True, masks=True)
    
    axes[idx].imshow(annotated_img)
    axes[idx].set_title(os.path.basename(img_path), fontsize=10)
    axes[idx].axis("off")  # Hide grid borders and coordinates

# Tight layout adjustments for notebook embedding
plt.tight_layout()
plt.show()